# Comprendre le RAG — cours pour débutant

Bienvenue ! Ce cours part de zéro et construit progressivement un mini-système RAG.

À la fin, tu sauras expliquer :

- pourquoi un LLM seul ne suffit pas toujours ;
- comment préparer et découper des documents ;
- ce que sont les embeddings et la recherche sémantique ;
- comment créer un prompt fondé sur des sources ;
- comment évaluer un RAG et réduire les hallucinations.

Le mini-corpus utilisé ici est **fictif et uniquement pédagogique**. Il ne contient aucun conseil juridique.

## 1. L'idée en une phrase

**RAG** signifie **Retrieval-Augmented Generation**, ou « génération augmentée par la recherche ».

Imagine un étudiant pendant un examen :

- un **LLM seul** répond avec ce qu'il a mémorisé ;
- un **RAG** commence par ouvrir les bons documents, puis répond en s'appuyant dessus.

Le RAG n'entraîne pas nécessairement un nouveau modèle. Il fournit au modèle un contexte pertinent au moment de la question.

![Comparaison entre un LLM seul et un système RAG](assets/llm_vs_rag.png)

## 2. Le pipeline complet

Un système RAG suit généralement cinq étapes :

1. l'utilisateur pose une question ;
2. le système cherche les passages les plus proches ;
3. il sélectionne les documents pertinents ;
4. il donne ces passages au LLM comme contexte ;
5. le LLM produit une réponse accompagnée de sources.

![Les cinq étapes d'un pipeline RAG](assets/rag_pipeline.png)

## 3. Notre mini-corpus

Pour comprendre le mécanisme, nous allons utiliser quatre petits documents inventés. Dans un vrai projet, ils seraient remplacés par des PDF officiels avec titre, date, URL et numéro de page.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
INDEX = DATA / "index"
for directory in (RAW, PROCESSED, INDEX):
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

In [ ]:
documents = [
    {
        "id": "DOC-A",
        "title": "Guide pédagogique des demandes",
        "page": 1,
        "text": "Une demande fictive doit contenir un formulaire signé et une copie du justificatif pédagogique.",
    },
    {
        "id": "DOC-B",
        "title": "Délais de traitement — exemple",
        "page": 3,
        "text": "Dans cet exemple fictif, le délai indicatif de traitement est de dix jours ouvrables.",
    },
    {
        "id": "DOC-C",
        "title": "Procédure de correction — exemple",
        "page": 2,
        "text": "Une erreur dans le dossier pédagogique peut être corrigée avec une demande écrite et le document rectifié.",
    },
    {
        "id": "DOC-D",
        "title": "Canaux de dépôt — exemple",
        "page": 4,
        "text": "Le dépôt fictif peut être effectué au guichet pédagogique ou sur la plateforme de démonstration.",
    },
]

for document in documents:
    print(f"{document['id']} — {document['title']} — page {document['page']}")

## 4. Pourquoi découper les documents ?

Un PDF peut contenir des centaines de pages. Envoyer tout le PDF au modèle serait lent, coûteux et souvent impossible à cause de la taille maximale du contexte.

On le découpe donc en **chunks**, c'est-à-dire en petits passages. Un léger chevauchement évite de couper une idée importante exactement entre deux chunks.

![Le chunking et les embeddings expliqués visuellement](assets/chunking_embeddings.png)

In [ ]:
def chunk_words(text: str, size: int = 10, overlap: int = 3) -> list[str]:
    words = text.split()
    step = max(1, size - overlap)
    return [" ".join(words[start:start + size]) for start in range(0, len(words), step)]

example = (
    "Un document très long doit être découpé en passages plus petits "
    "afin de retrouver seulement les informations utiles à la question."
)
chunk_words(example)

### Comment choisir la taille d'un chunk ?

Il n'existe pas de taille parfaite :

- trop petit : le passage perd son contexte ;
- trop grand : le passage mélange plusieurs sujets ;
- bon compromis : une idée juridique cohérente, avec ses métadonnées.

Dans le projet final, nous comparerons plusieurs tailles au lieu d'en choisir une au hasard.

## 5. Embeddings : représenter le sens avec des nombres

Un embedding est une liste de nombres représentant approximativement le sens d'un texte. Deux passages qui parlent de concepts proches ont généralement des vecteurs proches.

Exemple conceptuel :

- « délai de traitement » et « combien de jours faut-il attendre ? » doivent être proches ;
- « copie du justificatif » et « météo de demain » doivent être éloignés.

Nous commençons avec un petit TF-IDF écrit en Python standard pour voir le mécanisme sans télécharger de modèle lourd. Plus tard, `multilingual-e5-base` permettra une vraie recherche sémantique multilingue.

In [ ]:
import math
import re
from collections import Counter

def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

tokenized_documents = [tokenize(document["text"]) for document in documents]
vocabulary = sorted({token for tokens in tokenized_documents for token in tokens})
document_count = len(tokenized_documents)
document_frequency = {
    term: sum(term in tokens for tokens in tokenized_documents)
    for term in vocabulary
}
idf = {
    term: math.log((1 + document_count) / (1 + document_frequency[term])) + 1
    for term in vocabulary
}

def tfidf_vector(text: str) -> list[float]:
    counts = Counter(tokenize(text))
    total = max(1, sum(counts.values()))
    return [(counts[term] / total) * idf.get(term, 0.0) for term in vocabulary]

document_vectors = [tfidf_vector(document["text"]) for document in documents]
print("Documents :", len(document_vectors))
print("Dimensions du vocabulaire :", len(vocabulary))

## 6. Retrieval : retrouver les bons passages

Nous transformons la question avec le même vectoriseur, puis calculons la similarité cosinus. Un score élevé signifie que la question et le passage sont proches dans cet espace.

In [ ]:
def cosine(left: list[float], right: list[float]) -> float:
    dot = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    if left_norm == 0 or right_norm == 0:
        return 0.0
    return dot / (left_norm * right_norm)

def retrieve_tfidf(question: str, k: int = 2) -> list[dict]:
    query_vector = tfidf_vector(question)
    scores = [cosine(query_vector, vector) for vector in document_vectors]
    best_indices = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)[:k]
    return [
        {**documents[index], "score": float(scores[index])}
        for index in best_indices
    ]

question = "Quel est le délai de traitement ?"
results = retrieve_tfidf(question)
[(result["id"], round(result["score"], 3), result["text"]) for result in results]

### Dense, lexical ou hybride ?

- **Recherche lexicale (BM25/TF-IDF)** : excellente pour les mots exacts, numéros d'articles et expressions rares.
- **Recherche dense (embeddings)** : meilleure pour les synonymes, reformulations et questions multilingues.
- **Recherche hybride** : combine les deux ; c'est généralement notre choix pour les documents juridiques.

## 7. Construire le contexte et les citations

Le modèle ne doit pas recevoir uniquement le texte. Il faut aussi transmettre l'identifiant du document et la page pour générer des citations vérifiables.

In [ ]:
def format_context(results: list[dict]) -> str:
    return "\n\n".join(
        f"[Source {item['id']}, page {item['page']}]\n{item['text']}"
        for item in results
    )

context = format_context(results)
print(context)

## 8. Prompt contraint par les sources

Le prompt définit les règles du modèle. Pour limiter les hallucinations, il lui demande :

- d'utiliser uniquement les extraits fournis ;
- de citer ses affirmations ;
- de signaler quand les informations sont insuffisantes ;
- de ne jamais présenter la réponse comme un avis juridique.

In [ ]:
def build_prompt(question: str, context: str, language: str = "français") -> str:
    parts = [
        "Tu es un assistant de recherche documentaire.",
        f"Réponds en {language} uniquement à partir des sources fournies.",
        "Ajoute une citation [Source ID, page N] après chaque affirmation importante.",
        "Si les sources ne permettent pas de répondre, indique-le clairement.",
        "Ne fournis pas d'avis juridique professionnel.",
        f"QUESTION : {question}",
        f"SOURCES :\n{context}",
    ]
    return "\n\n".join(parts)

prompt = build_prompt(question, context)
print(prompt)

## 9. Un mini-RAG sans LLM

Avant de connecter un modèle génératif, on peut déjà créer une réponse extractive. Cette étape permet de tester le retrieval et les citations indépendamment du LLM.

In [ ]:
def extractive_answer(question: str, threshold: float = 0.05) -> str:
    retrieved = retrieve_tfidf(question, k=2)
    useful = [item for item in retrieved if item["score"] >= threshold]
    if not useful:
        return "Les documents disponibles ne permettent pas de répondre à cette question."
    lines = [
        f"- {item['text']} [Source {item['id']}, page {item['page']}]"
        for item in useful
    ]
    return "Passages trouvés :\n" + "\n".join(lines)

print(extractive_answer("Combien de jours faut-il attendre ?"))

## 10. Pourquoi le multilingue est plus difficile ?

La même question peut apparaître sous différentes formes :

- français : « Quel est le délai ? » ;
- arabe : « ما هي مدة المعالجة؟ » ;
- darija : « شحال خاصني نتسنى؟ ».

TF-IDF ne comprend pas naturellement que ces phrases ont un sens proche. Un modèle d'embeddings multilingue projette leurs significations dans le même espace vectoriel.

In [ ]:
# Cellule optionnelle : elle télécharge le modèle lors de la première exécution.
# from sentence_transformers import SentenceTransformer
# multilingual_encoder = SentenceTransformer("intfloat/multilingual-e5-base")
# examples = [
#     "query: Quel est le délai ?",
#     "query: ما هي مدة المعالجة؟",
#     "query: شحال خاصني نتسنى؟",
# ]
# vectors = multilingual_encoder.encode(examples, normalize_embeddings=True)
# vectors @ vectors.T

## 11. Évaluer un RAG

Une belle réponse ne prouve pas que le système fonctionne. Il faut mesurer au moins deux parties séparément.

### Retrieval

- `Recall@K` : le bon document apparaît-il parmi les K premiers ?
- `MRR` : à quelle position apparaît le premier bon document ?

### Génération

- fidélité : la réponse est-elle soutenue par les sources ?
- exactitude des citations : document et page sont-ils corrects ?
- complétude : la réponse couvre-t-elle les éléments utiles ?
- abstention : refuse-t-elle correctement lorsque la preuve manque ?

In [ ]:
test_questions = [
    {"question": "Quel est le délai ?", "expected": "DOC-B"},
    {"question": "Comment corriger une erreur ?", "expected": "DOC-C"},
    {"question": "Où déposer la demande ?", "expected": "DOC-D"},
]

correct = 0
for test in test_questions:
    top_document = retrieve_tfidf(test["question"], k=1)[0]["id"]
    is_correct = top_document == test["expected"]
    correct += int(is_correct)
    print(test["question"], "→", top_document, "✓" if is_correct else "✗")

print("Accuracy@1 pédagogique :", correct / len(test_questions))

## 12. Erreurs fréquentes à éviter

1. indexer des documents sans conserver leur provenance ;
2. utiliser des chunks trop grands ou trop petits sans évaluation ;
3. mesurer uniquement la qualité du texte généré ;
4. laisser le modèle répondre lorsque les sources sont insuffisantes ;
5. mélanger des versions anciennes et actuelles d'un texte ;
6. afficher une citation qui ne soutient pas réellement l'affirmation ;
7. considérer le RAG comme un remplacement d'un professionnel du droit.

## 13. Exercices

**Niveau 1** — Ajoute un cinquième document fictif puis vérifie qu'il peut être retrouvé.

**Niveau 2** — Modifie `k` et observe comment les résultats changent.

**Niveau 3** — Remplace TF-IDF par `multilingual-e5-base` et compare les trois langues.

**Niveau 4** — Ajoute BM25 et fusionne le score lexical avec le score dense.

**Niveau 5** — Branche un LLM local, impose les citations et crée des tests d'abstention.

## 14. Résumé final

Un RAG est composé de deux grandes phases :

- **indexation** : préparer les documents, créer les chunks et calculer les embeddings ;
- **question-réponse** : retrouver les bons chunks, construire le prompt et générer une réponse citée.

La qualité finale dépend souvent davantage du corpus, des métadonnées, du chunking et du retrieval que du choix du plus grand LLM.

**Prochaine étape dans ce dépôt :** appliquer exactement ce pipeline à un corpus de documents juridiques marocains officiels, avec évaluation en français, arabe et darija.